In [1]:
import os, json, time
from dotenv import load_dotenv
from pageindex import PageIndexClient
from groq import Groq
from langchain_groq import ChatGroq

d:\Data Science and Machine Learning\Data Science and Gen AI Data - 2026\Agentic AI\Codes\newenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

# Load API keys
page_index_api_key = os.getenv("PAGEINDEX_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")
model = os.getenv("LLM_MODEL")

In [3]:
pi_client = PageIndexClient(api_key=page_index_api_key)
groq_model = Groq(api_key=groq_api_key)
llm = ChatGroq(model=model, api_key=groq_api_key)

print(f"PageIndex Client is Ready")

PageIndex Client is Ready


In [4]:
from pypdf import PdfReader

PDF_PATH = r"D:\Data Science and Machine Learning\Data Science and Gen AI Data - 2026\Agentic AI\Codes\LangGraph Components\Vectorless RAG\Data Processing Steps in ML.pdf"

reader = PdfReader(PDF_PATH)

print("Number of pages:", len(reader.pages))

Number of pages: 42


In [5]:
PDF_PATH = r"D:\Data Science and Machine Learning\Data Science and Gen AI Data - 2026\Agentic AI\Codes\LangGraph Components\Vectorless RAG\Data Processing Steps in ML.pdf"

print(f"Uploading document: {PDF_PATH}")

result = pi_client.submit_document(PDF_PATH)

print(result)

doc_id = result["doc_id"]
print("Document ID:", doc_id)

Uploading document: D:\Data Science and Machine Learning\Data Science and Gen AI Data - 2026\Agentic AI\Codes\LangGraph Components\Vectorless RAG\Data Processing Steps in ML.pdf
{'doc_id': 'pi-cmt85o4cl02rw01p5ij5bjitk', 'name': 'Data Processing Steps in ML_2.pdf'}
Document ID: pi-cmt85o4cl02rw01p5ij5bjitk


In [6]:
# PageIndex builds tree asynchronously
# For a 50 page PDF, this typically takes 30-90 seconds

print("Building tree index...")

while True:
    status_result = pi_client.get_document(doc_id=doc_id)
    status = status_result.get("status")
    print(f"Status: {status}")

    if status.lower() == "completed":
        print(f"\nTree index is ready!")
        break
    elif status.lower() == "failed":
        print("Processing failed. Check your PDF format")
        break

    time.sleep(5)

Building tree index...
Status: processing
Status: processing
Status: processing
Status: completed

Tree index is ready!


In [7]:
# Inspect the Tree Structure
tree_result = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"Top level sections: {len(pageindex_tree)}\n")
print("RAW TREE FIRST NODE:\n")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

Top level sections: 1

RAW TREE FIRST NODE:

{
  "title": "DATA PROCESSING STEPS",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "This document outlines key machine learning data processing and modeling topics, including feature engineering, model optimization, performance evaluation, and the systematic steps required to solve machine learning problems.",
  "text": "# DATA PROCESSING STEPS\n\n![img-0.jpeg](img-0.jpeg)\n\nMachine Learning\n\n|  Sr. No | Topics | Sub-topics  |\n| --- | --- | --- |\n|  1 | OUTLIERS HANDLING |   |\n|  2 | CATEGORICAL ENCODING |   |\n|  3 | FEATURE SCALING TECHNIQUES |   |\n|  4 | FEATURE SELECTIONS AND EXTRACTIONS TECHNIQUES |   |\n|  5 | HYPERPARAMETER TUNING METHODS |   |\n|  6 | BIAS AND VARIANCE |   |\n|  7 | DATA DRIFT VS MODEL DRIFT |   |\n|  8 | HANDLING IMBALANCED DATA FOR CLASSIFICATION |   |\n|  9 | EVALUATION METRICS FOR REGRESSION & CLASSIFAICATION MODELS |   |\n|  10 | STEPS TO SOLVE ML PROBLEMS |   |\n|  |   |   |\n",
  "nodes":

In [8]:
# Pretty print the full tree
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview"""

    for node in nodes:
        prefix = " " * indent + ("^_" if indent > 0 else "")
        page = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']} (p.{page})")

        if node.get("nodes"):
            print_tree(node['nodes'], indent+1)

print("Full Document Structure")
print()
print_tree(pageindex_tree)

Full Document Structure

[0000] DATA PROCESSING STEPS (p.1)
 ^_[0001] OUTLIERS HANDLING (p.3)
  ^_[0002] Outliers Handling Methods (p.3)
   ^_[0003] 1. Z-Score: (p.3)
   ^_[0004] 2. IQR Method (Interquartile Range): (p.3)
   ^_[0005] 3. Isolation Forest: (p.4)
   ^_[0006] 4. Log Transformation: (p.4)
 ^_[0007] Categorical Data Encoding Techniques (p.7)
 ^_[0008] Feature Scaling Techniques (p.11)
  ^_[0009] 1. Absolute Maximum Scaling (p.11)
  ^_[0010] 2. Min-Max Scaling (p.12)
  ^_[0011] 3. Normalization (Vector Normalization) (p.13)
  ^_[0012] 4. Standardization (Z-Score Scaling) (p.13)
  ^_[0013] 5. Robust Scaling (p.14)
 ^_[0014] Data Sampling Methods (p.15)
 ^_[0015] Hyperparameter Tuning Methods (p.21)
  ^_[0016] Methods for Hyperparameter Tuning: (p.21)
  ^_[0017] 1. Manual Hyperparameter Tuning: (p.21)
  ^_[0018] 2. GridSearchCV: (p.21)
  ^_[0019] 3. RandomizedSearchCV: (p.22)
  ^_[0020] 4. Bayesian Optimization: (p.23)
 ^_[0021] *Bias and Variance* (p.25)
 ^_[0022] Data Drift v

In [9]:
# Count nodes
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n['nodes'])
    return total

total = count_nodes(pageindex_tree)
print(f"Total nodes in tree: {total}")

Total nodes in tree: 29


## **LLM Tree Search**

In [10]:
def llm_tree_search(
    query: str,
    tree: list,
    model: str = "openai/gpt-oss-120b"
) -> dict:

    def compress(nodes):
        out = []

        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title": n["title"],
                "page": n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]
            }

            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])

            out.append(entry)

        return out

    compressed_tree = compress(tree)

    prompt = f"""
    You are given a query and a document's tree structure.

    Your task is to identify which node IDs most likely contain
    the answer to the query.

    Query:
    {query}

    Document Tree:
    {json.dumps(compressed_tree, indent=2)}

    Return ONLY valid JSON:

    {{
        "thinking": "brief explanation of why these nodes are relevant",
        "node_list": ["node_id1", "node_id2"]
    }}
    """

    response = groq_model.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        response_format={"type": "json_object"}
    )

    return json.loads(
        response.choices[0].message.content
    )

In [11]:
# Test with a sample query 
query = "What is the syllabus covered in Hypeparameter tuning?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What is the syllabus covered in Hypeparameter tuning?

🧠 LLM Reasoning:
Nodes related to hyperparameter tuning are under node 0015 and its children which list the methods (manual, GridSearchCV, RandomizedSearchCV, Bayesian Optimization). These contain the syllabus of topics covered.

🎯 Selected Node IDs: ['0015', '0016', '0017', '0018', '0019', '0020']


## **Section 5: Full End-to-End RAG Pipeline**

**There are 3 steps:**

1. Tree Search → LLM picks relevant node_ids
2. Retrieve → Fetch the actual section content from those nodes
3. Generate → LLM writes a grounded answer with page citations

**What makes this better than vector RAG:**

* Retrieved content has titles + page numbers (traceable)
* LLM can cite exactly which section the answer comes from
* No hallucination from irrelevant chunks

In [12]:
# Helper: Find nodes by ID

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids"""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [13]:
# Generate answer from retrieved nodes

def generate_answer(query: str, nodes: list, model: str = "openai/gpt-oss-120b") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
    Answer the question using ONLY the provided context.
    For every claim you make, cite the section title and page number in parentheses.
    Be concise and precise.

    Question: {query}

    Context:
    {context}

    Answer:"""
    
    response = groq_model.chat.completions.create(
        model=model,
        messages=[
            {
            "role": "user", 
            "content": prompt
            }
                ]
    )
    
    return response.choices[0].message.content

In [14]:
# Complete Vectorless RAG function

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\nReasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [15]:
# Run the full pipeline 
answer = vectorless_rag(
    query="What are the topics covered in outliers handling?",
    tree=pageindex_tree
)

Query: What are the topics covered in outliers handling?

Reasoning: Nodes under the OUTLIERS HANDLING section list the specific methods used to handle outliers. Node 0002 outlines the main topics (the methods), and its child nodes 0003‑0006 provide the individual meth...
Retrieved node IDs: ['0002', '0003', '0004', '0005', '0006']
Sections found: ['Outliers Handling Methods', '1. Z-Score:', '2. IQR Method (Interquartile Range):', '3. Isolation Forest:', '4. Log Transformation:']

📝 Answer:
The outliers‑handling material covers the following topics:

- **Z‑Score method** – a statistical technique that flags points whose distance from the mean exceeds a chosen standard‑deviation threshold. (Section ‘1. Z‑Score:’, Page 3)  
- **IQR Method (Interquartile Range)** – a robust approach that identifies outliers outside Q₁ − 1.5·IQR or Q₃ + 1.5·IQR. (Section ‘2. IQR Method (Interquartile Range):’, Page 3)  
- **Isolation Forest** – an algorithm that isolates rare, different points quickly to d

---
## 📊 Section 9: Vector RAG vs PageIndex — Side-by-Side

### Architecture Comparison

| Aspect | Traditional Vector RAG | PageIndex (Vectorless RAG) |
|--------|------------------------|---------------------------|
| **Document prep** | Chunk into fixed pieces | Build hierarchical tree |
| **Indexing** | Embed each chunk | LLM reads structure |
| **Storage** | Vector database | JSON file |
| **Query processing** | Embed query → ANN search | LLM reasons over tree |
| **What's retrieved** | Flat anonymous chunks | Named sections + page refs |
| **Explainability** | ❌ Opaque similarity score | ✅ Traceable reasoning |
| **Domain expertise** | ❌ Needs embedding fine-tune | ✅ Add rules to prompt |
| **Infrastructure** | Pinecone / FAISS / ChromaDB | No vector DB needed |
| **Best for** | Short, diverse documents | Long, structured documents |
| **FinanceBench accuracy** | ~80% | **98.7%** |

### When to use which

**Use Vector RAG when:**
- Documents are short and varied (FAQs, product descriptions)
- Semantic paraphrase matching is important  
- You need sub-second retrieval on millions of documents

**Use PageIndex when:**
- Documents are long and professionally structured (reports, manuals, legal docs)
- You need traceable, cited answers
- Domain expertise should guide retrieval
- You want to avoid vector DB infrastructure
